# Mortality classification tasks - Charlson

In [ ]:
import yaml
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
import joblib
import os

# Load all models
ALL_MODELS = yaml.safe_load(open("../config.yml"))["models"]

# Filter for Charlson Mortality models
# We exclude 'survival' and 'los' to focus on the classification mortality tasks
relevant_keys = [k for k in ALL_MODELS.keys() if k.startswith("charlson_") and "los" not in k and "survival" not in k]
print(f"Models to train: {relevant_keys}")

# Variable to control saving of models
# Set to False if you do not want to save the models after evaluation
SAVE_MODELS = True

## Index-Specific Train/Test Splits

Create views of the train/test splits which have the comorbidities for each index present.

### Data Aggregation

The training data can be quite large which results in extremely large inputs for the models. For example, the MACSS has 100 comorbidities so the training data is a N x 100 matrix where N is the number of rows.

To improve the training time for our models, we can 'compress' the data and represent it by identifying each unique combination of comorbidities and the number of times it occurred.

For example, given the following row-level data:

| COMORB_1 | COMORB_2 | COMORB_3 |
| -------- | -------- | -------- |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    0     |
|    1     |    0     |    1     |
|    1     |    0     |    1     |

We would represent it as:

| COMORB_1 | COMORB_2 | COMORB_3 | N |
| -------- | -------- | -------- | - |
|    1     |    1     |    1     | 7 |
|    1     |    1     |    0     | 1 |
|    1     |    0     |    1     | 2 |

In [ ]:
data_dict = {
    "CONFIGS": {},
    "TRAINING_DATASETS": {},
    "TESTING_DATASETS": {},
    "TRAINING_AGGREGATIONS": {},
    "TESTING_AGGREGATIONS": {},
    "MODELS":{}
}

In [ ]:
# Load configs into data_dict
for key in relevant_keys:
    data_dict["CONFIGS"][key] = ALL_MODELS[key]

In [ ]:
# Load training and testing datasets into data_dict
for item in data_dict["CONFIGS"]:
    data_dict["TRAINING_DATASETS"][item] = pd.read_csv(f"../datasets/{data_dict['CONFIGS'][item]['dataset_training']}")
    data_dict["TESTING_DATASETS"][item] = pd.read_csv(f"../datasets/{data_dict['CONFIGS'][item]['dataset_testing']}")


In [ ]:
# Load the aggregated datasets into data_dict
for item in data_dict["CONFIGS"]:
    data_dict["TRAINING_AGGREGATIONS"][item] = data_dict["TRAINING_DATASETS"][item].groupby(list(data_dict["TRAINING_DATASETS"][item].columns),dropna=False).size().reset_index(name='N')
    data_dict["TESTING_AGGREGATIONS"][item] = data_dict["TESTING_DATASETS"][item].groupby(list(data_dict["TESTING_DATASETS"][item].columns),dropna=False).size().reset_index(name='N')

In [ ]:
for key in relevant_keys:
    MODEL_CONFIG = data_dict["CONFIGS"][key]
    print(f"\nProcessing {key} (Target: {MODEL_CONFIG['class']})...")
    
    # Data Aggregation
    charls_training_agg = data_dict["TRAINING_AGGREGATIONS"][key]

    # Define input columns
    input_cols = [i for i in charls_training_agg.columns if "C_" in i]
    
    # Train
    clf = LogisticRegression(penalty=None)
    clf.fit(charls_training_agg[input_cols], charls_training_agg[MODEL_CONFIG["class"]], sample_weight=charls_training_agg['N'])

    data_dict["MODELS"][key] = clf

In [ ]:
for key in relevant_keys:
    MODEL_CONFIG = data_dict["CONFIGS"][key]
    print(f"\nProcessing {key} (Target: {MODEL_CONFIG['class']})...")
    
    # Data Aggregation
    charls_training_agg = data_dict["TRAINING_AGGREGATIONS"][key]
    charls_testing_agg = data_dict["TESTING_AGGREGATIONS"][key]
    
    # Define input columns
    input_cols = [i for i in charls_training_agg.columns if "C_" in i]

    clf = data_dict["MODELS"][key]

    # Predict
    y_train_pred_proba = clf.predict_proba(charls_training_agg[input_cols])
    y_train_true = charls_training_agg[MODEL_CONFIG["class"]]

    y_test_pred_proba = clf.predict_proba(charls_testing_agg[input_cols])
    y_test_true = charls_testing_agg[MODEL_CONFIG["class"]]

    # Metrics
    fpr_train, tpr_train, _ = roc_curve(y_train_true, y_train_pred_proba[:,1], sample_weight=charls_training_agg['N'])
    print(f"  Training AUC: {auc(fpr_train, tpr_train):.4f}")

    fpr_test, tpr_test, _ = roc_curve(y_test_true, y_test_pred_proba[:,1], sample_weight=charls_testing_agg['N'])
    print(f"  Test AUC: {auc(fpr_test, tpr_test):.4f}")

    if SAVE_MODELS:
        # Save the model
        if not os.path.exists('../models'):
            os.makedirs('../models')

        save_path = f'../models/mort_charlson_{MODEL_CONFIG["class"]}.joblib'
        joblib.dump(clf, save_path)
        print(f"  Model saved to {save_path}")